# Part 2: Generation and Evaluation

Generates images with the base model and both LoRA runs, then evaluates them (2.3 Image Generation, 2.4 Evaluation, Safety & Documentation). This notebook does the GPU work that `p2_finetune.ipynb` (data prep and training) does not -- it loads the two adapters trained there from Drive.

**Not yet run on a GPU.** Drafted while the two training runs were in progress; only the pieces that need no GPU (readers, the grid layout, the blind-pairs sheet/answer-key split and its scoring, the metadata writer) were tested, with tiny fake data, on a Mac.

## Setup

In [ ]:
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})


In [ ]:
# lab rule: all required Part 2 work MUST run on one 16 GB GPU (Kaggle T4/P100 or Colab T4); fp16 throughout
ALLOW_OTHER_GPU = False
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
is_t4 = "T4" in gpu_name
print("GPU:", gpu_name, "| T4:", is_t4)
if not is_t4 and not ALLOW_OTHER_GPU:
    raise RuntimeError(f"this notebook requires a Colab T4 GPU; got {gpu_name!r}. Set ALLOW_OTHER_GPU to override.")


In [ ]:
import collections, hashlib, io, json as j
from pathlib import Path
from PIL import Image
from IPython.display import display

device = "cuda" if torch.cuda.is_available() else "cpu"
DRIVE_ROOT = Path("/content/drive/MyDrive/DATA266-Lab2/p2")
DATA_DIR = Path("/content/p2_data")  # rebuilt locally here too -- see "Rebuild the data split" below
GEN_DIR = DRIVE_ROOT / "generations"
GEN_DIR.mkdir(parents=True, exist_ok=True)


### Installs (pinned versions)

In [ ]:
import importlib.metadata as im

# torchao pinned for the same reason as in p2_finetune.ipynb: peft 0.21.2 raises (not just skips)
# when an installed torchao is older than it requires, and loading a LoRA adapter goes through the
# same peft dispatch path as training does
PINS = {"diffusers": "0.42.1", "peft": "0.21.2", "accelerate": "1.15.0", "datasets": "4.8.5", "torchao": "0.18.0",
        "torchmetrics": None, "torch-fidelity": None, "lpips": None, "open_clip_torch": None}
to_install = [f"{pkg}=={ver}" if ver else pkg for pkg, ver in PINS.items()]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + to_install, check=True)
for pkg in ("diffusers", "peft", "accelerate", "datasets", "torchao", "torchmetrics", "torch-fidelity", "lpips", "open_clip_torch"):
    print(f"{pkg}: {im.version(pkg)}")


### Google Drive and Hugging Face

In [ ]:
# Drive is mounted BEFORE this notebook runs, from the CLI: `colab drivemount -s <session>`
assert Path("/content/drive/MyDrive").exists(), "Drive not mounted -- run 'colab drivemount -s <session>' first"
print("Drive already mounted at /content/drive")


In [ ]:
# Hugging Face auth is set up BEFORE this notebook runs, from the CLI: copy the local token file with
# `colab upload -s <session> ~/.cache/huggingface/token /root/.cache/huggingface/token`. The token is only
# ever read by huggingface_hub from that file -- never read, printed or logged by this notebook.
from huggingface_hub import HfApi
print("HF username:", HfApi().whoami()["name"])


## Fixed evaluation prompts

The same 20 prompts as `p2_finetune.ipynb` (written before training). Embedded here for the same reason: the Colab VM cannot read the repo.

In [ ]:
PROMPTS = [{'id': 'P01', 'type': 'in_domain', 'prompt': 'a red fish with a gold crown on its head', 'test_image_id': 35}, {'id': 'P02', 'type': 'in_domain', 'prompt': 'a very cute looking pokemon with big eyes', 'test_image_id': 44}, {'id': 'P03', 'type': 'in_domain', 'prompt': 'a bird with a long beak flying through the air', 'test_image_id': 97}, {'id': 'P04', 'type': 'in_domain', 'prompt': 'a yellow and white cartoon character with a red eye', 'test_image_id': 100}, {'id': 'P05', 'type': 'in_domain', 'prompt': 'a gray and yellow dragon with large horns', 'test_image_id': 392}, {'id': 'P06', 'type': 'in_domain', 'prompt': 'a robot with a yellow helmet and green legs', 'test_image_id': 491}, {'id': 'P07', 'type': 'in_domain', 'prompt': 'a cartoon picture of a bear holding a baseball bat', 'test_image_id': 19}, {'id': 'P08', 'type': 'in_domain', 'prompt': 'a black and white picture of a cat wearing a witches hat', 'test_image_id': 639}, {'id': 'P09', 'type': 'in_domain', 'prompt': 'a very cute looking animal with big ears', 'test_image_id': 418}, {'id': 'P10', 'type': 'in_domain', 'prompt': 'a cartoon character with pink hair and a pom pom on top of it', 'test_image_id': 757}, {'id': 'P11', 'type': 'out_of_domain', 'prompt': 'a red double-decker bus on a rainy london street'}, {'id': 'P12', 'type': 'out_of_domain', 'prompt': 'a bowl of ramen with chopsticks on a wooden table'}, {'id': 'P13', 'type': 'out_of_domain', 'prompt': 'a lighthouse on a rocky coast at sunset'}, {'id': 'P14', 'type': 'out_of_domain', 'prompt': 'an astronaut riding a bicycle on the moon'}, {'id': 'P15', 'type': 'out_of_domain', 'prompt': 'a portrait of an old fisherman wearing a yellow raincoat'}, {'id': 'P16', 'type': 'compositional', 'prompt': 'two cartoon creatures, one red and one blue, standing side by side'}, {'id': 'P17', 'type': 'compositional', 'prompt': 'a small green creature sitting on top of a large orange creature'}, {'id': 'P18', 'type': 'compositional', 'prompt': 'three yellow creatures in a row, the middle one wearing a hat'}, {'id': 'P19', 'type': 'compositional', 'prompt': 'a blue fish-like creature on the left and a purple bird on the right'}, {'id': 'P20', 'type': 'compositional', 'prompt': 'a white cat-like creature holding a red ball under a tree'}]

print(len(PROMPTS), "fixed prompts embedded |", collections.Counter(p["type"] for p in PROMPTS))

_eval_dir = Path("eval")
_f1, _f2 = _eval_dir / "p2_prompts_manav.json", _eval_dir / "p2_prompts_manav_ood_comp.json"
if _f1.exists() and _f2.exists():
    _from_files = j.loads(_f1.read_text()) + j.loads(_f2.read_text())
    assert PROMPTS == _from_files, "embedded PROMPTS does not match eval/p2_prompts_manav*.json"
    print("embedded prompts match eval/p2_prompts_manav*.json")
else:
    print("eval files not found here (expected on the Colab VM) -- skipping the equality check")


### Rebuild the data split

Reuses the exact seed-266 split from `p2_finetune.ipynb` (same dedup, same split function) so the test images used for KID and the memorization check match training exactly. Saved locally again -- the Colab VM's own disk never persists between sessions.

In [ ]:
from datasets import load_dataset, Image as HFImage
import imagehash

ds = load_dataset("reach-vb/pokemon-blip-captions", split="train").cast_column("image", HFImage(decode=False))

PHASH_MAX_HAMMING = 4


def open_image(data):
    Image.open(io.BytesIO(data)).verify()
    img = Image.open(io.BytesIO(data))
    img.load()
    return img.convert("RGB")


corrupted, exact, near, keep = [], [], [], []
seen_pixels, seen_phash = {}, []
for i in range(len(ds)):
    try:
        img = open_image(ds[i]["image"]["bytes"])
    except Exception:
        corrupted.append(i)
        continue
    pixel_key = hashlib.md5(img.tobytes()).hexdigest()
    if pixel_key in seen_pixels:
        exact.append((seen_pixels[pixel_key], i))
        continue
    seen_pixels[pixel_key] = i
    ph = imagehash.phash(img)
    dup = next((j for j, h in seen_phash if ph - h <= PHASH_MAX_HAMMING), None)
    if dup is not None:
        near.append((dup, i))
        continue
    seen_phash.append((i, ph))
    keep.append(i)


def split(ids, seed=SEED, test_frac=0.1):
    ids = sorted(ids)
    random.Random(seed).shuffle(ids)
    n_test = round(len(ids) * test_frac)
    return sorted(ids[n_test:]), sorted(ids[:n_test])


train_ids, test_ids = split(keep)
print("rows after cleaning:", len(keep), "| train:", len(train_ids), "| test:", len(test_ids))


def to_512(img):
    s = min(img.size)
    left, top = (img.width - s) // 2, (img.height - s) // 2
    return img.crop((left, top, left + s, top + s)).resize((512, 512), Image.LANCZOS)


for name, ids in [("train", train_ids), ("test", test_ids)]:
    out = DATA_DIR / name
    out.mkdir(parents=True, exist_ok=True)
    for i in ids:
        file_name = "%04d.png" % i
        if not (out / file_name).exists():
            to_512(open_image(ds[i]["image"]["bytes"])).save(out / file_name)
print("train/test images on local disk:", len(list((DATA_DIR / "train").glob("*.png"))),
      len(list((DATA_DIR / "test").glob("*.png"))))


## 2.3 Image Generation

### Load the base model and both LoRA adapters

One pipeline instance, LoRA toggled on and off, to avoid loading three separate UNets into 16 GB.

In [ ]:
from diffusers import StableDiffusionPipeline

BASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
RUN_DIRS = {"run_r8": DRIVE_ROOT / "runs" / "run_r8", "run_r32": DRIVE_ROOT / "runs" / "run_r32"}

# no safety_checker kwarg here on purpose: leaving it unset keeps the pipeline's own default NSFW checker on
pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)
pipe = pipe.to(device)
# the pipeline's own default scheduler (PNDM for SD 1.5) is kept fixed across every model and seed below
NUM_INFERENCE_STEPS = 30
GUIDANCE_SCALE = 7.5
print("scheduler:", pipe.scheduler.__class__.__name__, "| steps:", NUM_INFERENCE_STEPS, "| guidance:", GUIDANCE_SCALE)
print("safety checker on:", pipe.safety_checker is not None)

_loaded_lora = None


def set_model(name):
    global _loaded_lora
    if name == _loaded_lora:
        return
    if _loaded_lora is not None:
        pipe.unload_lora_weights()
    if name != "base":
        pipe.load_lora_weights(str(RUN_DIRS[name]), weight_name="pytorch_lora_weights.safetensors")
    _loaded_lora = name


### Generate 120 images (base, run_r8, run_r32 x 20 prompts x seeds 0, 1)

In [ ]:
records = []
metadata_path = GEN_DIR / "metadata.jsonl"
existing = set()
if metadata_path.exists():
    existing = {(r["model"], r["prompt_id"], r["seed"]) for r in (j.loads(l) for l in metadata_path.read_text().splitlines())}

for model_name in ("base", "run_r8", "run_r32"):
    set_model(model_name)
    for p in PROMPTS:
        for seed in (0, 1):
            if (model_name, p["id"], seed) in existing:
                continue
            file_name = f"{model_name}_{p['id']}_s{seed}.png"
            g = torch.Generator(device=device).manual_seed(seed)
            out = pipe(p["prompt"], num_inference_steps=NUM_INFERENCE_STEPS, guidance_scale=GUIDANCE_SCALE,
                       height=512, width=512, generator=g, output_type="pil")
            out.images[0].save(GEN_DIR / file_name)
            nsfw = bool(out.nsfw_content_detected[0]) if getattr(out, "nsfw_content_detected", None) else False
            records.append({"file_name": file_name, "prompt_id": p["id"], "type": p["type"],
                             "prompt": p["prompt"], "seed": seed, "model": model_name, "nsfw": nsfw})

with open(metadata_path, "a") as f:
    for r in records:
        f.write(j.dumps(r) + "\n")

all_records = [j.loads(l) for l in metadata_path.read_text().splitlines()]
print(len(records), "images generated this run |", len(all_records), "total |",
      sum(r["nsfw"] for r in all_records), "flagged NSFW")


### Side-by-side grids for 8 prompts (seed-266 draw)

In [ ]:
GRID_PROMPT_IDS = random.Random(266).sample([p["id"] for p in PROMPTS], 8)
print("grid prompts:", GRID_PROMPT_IDS)


def make_grid(prompt_id, seed=0):
    imgs = [Image.open(GEN_DIR / f"{m}_{prompt_id}_s{seed}.png") for m in ("base", "run_r8", "run_r32")]
    w, h = imgs[0].size
    sheet = Image.new("RGB", (w * 3 + 20, h), "white")
    for i, im in enumerate(imgs):
        sheet.paste(im, (i * (w + 10), 0))
    return sheet


for pid in GRID_PROMPT_IDS:
    grid = make_grid(pid)
    grid.save(GEN_DIR / f"grid_{pid}.png")
    display(grid)


### Upload the 120 images to a Hugging Face dataset

In [ ]:
GEN_DATASET_REPO = "itzmnv/data266-lab2-sd15-pokemon-generations"
api = HfApi()
api.create_repo(GEN_DATASET_REPO, repo_type="dataset", exist_ok=True, private=False)
api.upload_folder(repo_id=GEN_DATASET_REPO, repo_type="dataset", folder_path=str(GEN_DIR),
                   allow_patterns=["*.png", "metadata.jsonl"])
print(f"https://huggingface.co/datasets/{GEN_DATASET_REPO}")


## 2.4 Evaluation, Safety & Documentation

### Primary: KID and CLIP score

KID uses a subset of at most 20 test images against the in-domain generations -- with this few images KID is noisy; treat it as a rough signal, not a precise number.

In [ ]:
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.multimodal.clip_score import CLIPScore
import torchvision.transforms.functional as TF
import pandas as pd

KID_SUBSET = 20


def to_uint8_tensor(path, size=299):
    img = Image.open(path).convert("RGB").resize((size, size))
    return TF.pil_to_tensor(img)


test_paths = [DATA_DIR / "test" / ("%04d.png" % i) for i in test_ids[:KID_SUBSET]]
print(f"KID test subset: {len(test_paths)} images (KID is noisy with this few -- a rough signal only)")

kid_results = {}
for model_name in ("run_r8", "run_r32"):
    in_domain_ids = [p["id"] for p in PROMPTS if p["type"] == "in_domain"]
    gen_paths = [GEN_DIR / f"{model_name}_{pid}_s{s}.png" for pid in in_domain_ids for s in (0, 1)][:KID_SUBSET]
    n = min(KID_SUBSET, len(gen_paths), len(test_paths))
    kid = KernelInceptionDistance(subset_size=n, normalize=False)
    kid.update(torch.stack([to_uint8_tensor(p) for p in test_paths[:n]]), real=True)
    kid.update(torch.stack([to_uint8_tensor(p) for p in gen_paths[:n]]), real=False)
    mean, std = kid.compute()
    kid_results[model_name] = {"kid_mean": mean.item(), "kid_std": std.item()}
    print(f"{model_name}: KID = {mean.item():.4f} +/- {std.item():.4f} (n={n})")

clip_metric = CLIPScore(model_name_or_path="openai/clip-vit-large-patch14").to(device)
all_records = [j.loads(l) for l in (GEN_DIR / "metadata.jsonl").read_text().splitlines()]
clip_rows = []
for r in all_records:
    img = TF.pil_to_tensor(Image.open(GEN_DIR / r["file_name"]).convert("RGB")).to(device)
    score = clip_metric(img, r["prompt"]).item()
    clip_rows.append({**r, "clip_score": score})
CLIP_DF = pd.DataFrame(clip_rows)
CLIP_DF.to_csv(DRIVE_ROOT / "clip_scores.csv", index=False)
print(CLIP_DF.groupby(["model", "type"])["clip_score"].mean().to_string())


### Secondary: LPIPS diversity (between seeds 0 and 1, per prompt and model)

In [ ]:
import lpips

lpips_fn = lpips.LPIPS(net="alex").to(device)


def to_lpips_tensor(path):
    img = Image.open(path).convert("RGB").resize((512, 512))
    t = TF.pil_to_tensor(img).float() / 127.5 - 1
    return t.unsqueeze(0).to(device)


lpips_rows = []
for model_name in ("base", "run_r8", "run_r32"):
    for p in PROMPTS:
        a = to_lpips_tensor(GEN_DIR / f"{model_name}_{p['id']}_s0.png")
        b = to_lpips_tensor(GEN_DIR / f"{model_name}_{p['id']}_s1.png")
        with torch.no_grad():
            d = lpips_fn(a, b).item()
        lpips_rows.append({"model": model_name, "prompt_id": p["id"], "type": p["type"], "lpips": d})
LPIPS_DF = pd.DataFrame(lpips_rows)
LPIPS_DF.to_csv(DRIVE_ROOT / "lpips_diversity.csv", index=False)
print(LPIPS_DF.groupby("model")["lpips"].mean().to_string())


### Human evaluation: blinded pairs

25 pairs (final run vs base), left/right randomized by seed 266. The rating sheet is genuinely blank -- the answer key (which side is the final run) is saved to a separate file, never shown to the rater.

In [ ]:
BLIND_N = 25
FINAL_MODEL = "run_r32"  # provisional pick; the real final choice is made after this evaluation runs

pairs_source = [(p["id"], s) for p in PROMPTS for s in (0, 1)]
rng = random.Random(266)
rng.shuffle(pairs_source)
blind_source = pairs_source[:BLIND_N]

blind_records = []
for i, (pid, seed) in enumerate(blind_source, 1):
    left_is_final = rng.random() < 0.5
    left_model = FINAL_MODEL if left_is_final else "base"
    right_model = "base" if left_is_final else FINAL_MODEL
    blind_records.append({
        "pair_id": i, "prompt_id": pid, "seed": seed,
        "left_file": f"{left_model}_{pid}_s{seed}.png", "right_file": f"{right_model}_{pid}_s{seed}.png",
        "left_is_final": left_is_final,
    })

RATING_SHEET = pd.DataFrame([{"pair_id": r["pair_id"], "prompt_id": r["prompt_id"], "left_file": r["left_file"],
                              "right_file": r["right_file"], "preferred (L/R)": ""} for r in blind_records])
ANSWER_KEY = pd.DataFrame([{"pair_id": r["pair_id"], "left_is_final": r["left_is_final"]} for r in blind_records])

RATING_SHEET_PATH = DRIVE_ROOT / "blind_rating_sheet.csv"
ANSWER_KEY_PATH = DRIVE_ROOT / "blind_answer_key.csv"
RATING_SHEET.to_csv(RATING_SHEET_PATH, index=False)
ANSWER_KEY.to_csv(ANSWER_KEY_PATH, index=False)
print("blank rating sheet:", RATING_SHEET_PATH)
print("answer key (not for the rater):", ANSWER_KEY_PATH)


In [ ]:
def score_blind_ratings(rating_sheet_path, answer_key_path):
    ratings = pd.read_csv(rating_sheet_path)
    key = pd.read_csv(answer_key_path)
    merged = ratings.merge(key, on="pair_id")
    answered = merged[merged["preferred (L/R)"].isin(["L", "R"])].copy()
    if len(answered) == 0:
        return {"n_rated": 0, "win_rate_final": None}
    answered["picked_final"] = (answered["preferred (L/R)"] == "L") == answered["left_is_final"]
    return {"n_rated": len(answered), "win_rate_final": float(answered["picked_final"].mean())}


def agreement_rate(sheet_a_path, sheet_b_path):
    a, b = pd.read_csv(sheet_a_path), pd.read_csv(sheet_b_path)
    merged = a.merge(b, on="pair_id", suffixes=("_a", "_b"))
    both = merged[merged["preferred (L/R)_a"].isin(["L", "R"]) & merged["preferred (L/R)_b"].isin(["L", "R"])]
    return float((both["preferred (L/R)_a"] == both["preferred (L/R)_b"]).mean()) if len(both) else None


# NOTE: rate the blank sheet yourself (open blind_rating_sheet.csv, fill "preferred (L/R)", re-upload or
# edit in place on Drive), then re-run this cell -- it reads whatever is in the sheet, nothing is fabricated here
print(score_blind_ratings(RATING_SHEET_PATH, ANSWER_KEY_PATH))


### Memorization check

CLIP image similarity of 10 in-domain generations against every training image, for each run.

In [ ]:
import open_clip

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
clip_model = clip_model.to(device).eval()


def clip_image_embed(path):
    img = clip_preprocess(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = clip_model.encode_image(img)
    return (emb / emb.norm(dim=-1, keepdim=True)).cpu()


MEMORIZATION_N = 10
mem_prompt_ids = random.Random(266).sample([p["id"] for p in PROMPTS if p["type"] == "in_domain"], MEMORIZATION_N)
train_embeds = {i: clip_image_embed(DATA_DIR / "train" / ("%04d.png" % i)) for i in train_ids}

memorization_rows = []
for model_name in ("run_r8", "run_r32"):
    print(f"--- memorization check: {model_name} ---")
    for pid in mem_prompt_ids:
        gen_embed = clip_image_embed(GEN_DIR / f"{model_name}_{pid}_s0.png")
        sims = {i: (gen_embed @ train_embeds[i].T).item() for i in train_ids}
        top3 = sorted(sims.items(), key=lambda x: -x[1])[:3]
        print(f"  {pid}: closest train images {top3}")
        memorization_rows.append({"model": model_name, "prompt_id": pid, "top3": top3})


### Failure modes

Two example generations likely to show a failure mode (an out-of-domain and a compositional prompt). Notes are left blank below for a human to fill in after looking -- nothing here is fabricated.

In [ ]:
for pid in ("P11", "P16"):
    prompt = next(p["prompt"] for p in PROMPTS if p["id"] == pid)
    img = Image.open(GEN_DIR / f"run_r32_{pid}_s0.png")
    print(pid, "--", prompt)
    display(img)
    print("Failure mode noted: ___ (fill in after looking)\n")


### License

SD 1.5 is licensed under CreativeML OpenRAIL-M. The Pokemon BLIP Captions data is CC BY-NC-SA 4.0, and Pokemon characters are copyrighted, so this model, its adapters and every generated image are for **non-commercial research only**.

### NSFW filter

In [ ]:
all_records = [j.loads(l) for l in (GEN_DIR / "metadata.jsonl").read_text().splitlines()]
n_flagged = sum(r["nsfw"] for r in all_records)
print(f"safety checker: on | images flagged NSFW: {n_flagged} of {len(all_records)}")


## Model card -- after evaluation

Written once the results above pick a final run between rank 8 and rank 32. Not yet, per instruction.